In [1]:
import sqlite3

import numpy as np
import pandas as pd

from scipy.stats import (
    binomtest,
    chi2_contingency,
    norm
)

In [2]:
conn = sqlite3.connect(
    "../data/processed/olist.db"
)

analysis_df = pd.read_sql_query(
    """
    SELECT
        f.*,
        y.repeat_90d_after_1h

    FROM first_purchase_feature_base AS f

    JOIN customer_repeat_90d_base AS y
        ON f.customer_unique_id
            = y.customer_unique_id

    WHERE
        y.is_eligible_90d = 1
    """,
    conn
)

conn.close()

In [3]:
print(
    analysis_df.shape
)

print(
    analysis_df[
        "repeat_90d_after_1h"
    ]
    .value_counts()
)

(78505, 23)
repeat_90d_after_1h
0    77423
1     1082
Name: count, dtype: int64


# Basket Size 검정

Single Item
70,197명 / Positive 925명 / 1.32%

Multi Item
7,730명 / Positive 149명 / 1.93%

In [4]:
item_analysis_df = (
    analysis_df[
        analysis_df[
            "first_item_count"
        ].notna()
    ]
    .copy()
)

item_analysis_df[
    "multi_item_order"
] = (
    item_analysis_df[
        "first_item_count"
    ] >= 2
)

item_group_df = (
    item_analysis_df
    .groupby(
        "multi_item_order"
    )
    ["repeat_90d_after_1h"]
    .agg(
        [
            "count",
            "sum",
            "mean"
        ]
    )
)

print(item_group_df)

                  count  sum      mean
multi_item_order                      
False             70197  925  0.013177
True               7730  149  0.019276


In [5]:
# Wilson 95% CI
single_n = (
    item_group_df.loc[
        False,
        "count"
    ]
)

single_repeat = (
    item_group_df.loc[
        False,
        "sum"
    ]
)

multi_n = (
    item_group_df.loc[
        True,
        "count"
    ]
)

multi_repeat = (
    item_group_df.loc[
        True,
        "sum"
    ]
)


single_ci = (
    binomtest(
        int(single_repeat),
        int(single_n)
    )
    .proportion_ci(
        confidence_level=0.95,
        method="wilson"
    )
)

multi_ci = (
    binomtest(
        int(multi_repeat),
        int(multi_n)
    )
    .proportion_ci(
        confidence_level=0.95,
        method="wilson"
    )
)

print(
    "Single-Item 95% CI:",
    single_ci
)

print(
    "Multi-Item 95% CI:",
    multi_ci
)

Single-Item 95% CI: ConfidenceInterval(low=np.float64(0.012359875472718993), high=np.float64(0.014047805978053133))
Multi-Item 95% CI: ConfidenceInterval(low=np.float64(0.016440767895406192), high=np.float64(0.02258789081551608))


In [6]:
# Two-Proportion Z-Test
single_rate = (
    single_repeat
    / single_n
)

multi_rate = (
    multi_repeat
    / multi_n
)

pooled_rate = (
    single_repeat
    + multi_repeat
) / (
    single_n
    + multi_n
)

standard_error = np.sqrt(
    pooled_rate
    * (1 - pooled_rate)
    * (
        1 / single_n
        + 1 / multi_n
    )
)

z_score = (
    multi_rate
    - single_rate
) / standard_error

basket_p_value = (
    2
    * (
        1
        - norm.cdf(
            abs(z_score)
        )
    )
)

absolute_difference = (
    multi_rate
    - single_rate
)

relative_risk = (
    multi_rate
    / single_rate
)

print(
    "Single:",
    f"{single_rate:.2%}"
)

print(
    "Multi:",
    f"{multi_rate:.2%}"
)

print(
    "Difference:",
    f"{absolute_difference * 100:.2f}%p"
)

print(
    "Relative Risk:",
    round(
        relative_risk,
        3
    )
)

print(
    "Z:",
    round(
        z_score,
        4
    )
)

print(
    "p-value:",
    basket_p_value
)

Single: 1.32%
Multi: 1.93%
Difference: 0.61%p
Relative Risk: 1.463
Z: 4.3649
p-value: 1.2719034494734771e-05


# First Order Value

Q1  1.43%
Q2  1.36%
Q3  1.33%
Q4  1.39%

In [7]:
# Chi-Square
order_value_df = (
    analysis_df[
        analysis_df[
            "first_order_value"
        ].notna()
    ]
    .copy()
)

order_value_df[
    "order_value_group"
] = pd.qcut(
    order_value_df[
        "first_order_value"
    ],
    q=4,
    labels=[
        "Q1",
        "Q2",
        "Q3",
        "Q4"
    ]
)

order_value_table = pd.crosstab(
    order_value_df[
        "order_value_group"
    ],
    order_value_df[
        "repeat_90d_after_1h"
    ]
)

chi2, order_value_p_value, dof, expected = (
    chi2_contingency(
        order_value_table
    )
)

print(
    order_value_table
)

print(
    "p-value:",
    order_value_p_value
)

repeat_90d_after_1h      0    1
order_value_group              
Q1                   19251  279
Q2                   19222  266
Q3                   19175  258
Q4                   19205  271
p-value: 0.8541367844002381


In [8]:
# Cramer V
n = (
    order_value_table
    .to_numpy()
    .sum()
)

order_value_cramers_v = np.sqrt(
    chi2
    / (
        n
        * (
            min(
                order_value_table.shape
            )
            - 1
        )
    )
)

print(
    "Cramer's V:",
    round(
        order_value_cramers_v,
        4
    )
)

Cramer's V: 0.0032


# Payment Type

credit_card   817 / 59,297 = 1.38%
boleto        208 / 15,900 = 1.31%
voucher        44 / 2,481  = 1.77%
debit_card     13 / 826    = 1.57%

In [9]:
payment_df = (
    analysis_df[
        analysis_df[
            "primary_payment_type"
        ].notna()
    ]
)

payment_table = pd.crosstab(
    payment_df[
        "primary_payment_type"
    ],
    payment_df[
        "repeat_90d_after_1h"
    ]
)

chi2, payment_p_value, dof, expected = (
    chi2_contingency(
        payment_table
    )
)

n = (
    payment_table
    .to_numpy()
    .sum()
)

payment_cramers_v = np.sqrt(
    chi2
    / (
        n
        * (
            min(
                payment_table.shape
            )
            - 1
        )
    )
)

print(
    "Payment p-value:",
    payment_p_value
)

print(
    "Payment Cramer's V:",
    round(
        payment_cramers_v,
        4
    )
)

Payment p-value: 0.3008313374953709
Payment Cramer's V: 0.0068


# Product Category

n >= 500

In [10]:
category_count = (
    analysis_df[
        "primary_category"
    ]
    .value_counts()
)

large_category = (
    category_count[
        category_count >= 500
    ]
    .index
)

category_analysis_df = (
    analysis_df[
        analysis_df[
            "primary_category"
        ]
        .isin(
            large_category
        )
    ]
)

category_table = pd.crosstab(
    category_analysis_df[
        "primary_category"
    ],
    category_analysis_df[
        "repeat_90d_after_1h"
    ]
)

chi2, category_p_value, dof, expected = (
    chi2_contingency(
        category_table
    )
)

n = (
    category_table
    .to_numpy()
    .sum()
)

category_cramers_v = np.sqrt(
    chi2
    / (
        n
        * (
            min(
                category_table.shape
            )
            - 1
        )
    )
)

print(
    "Category p-value:",
    category_p_value
)

print(
    "Category Cramer's V:",
    round(
        category_cramers_v,
        4
    )
)

Category p-value: 2.615256334705229e-10
Category Cramer's V: 0.0362
